# Kinetics coverage: BRENDA and SABIO-RK

Two summaries of the local raw files in `data/proteolysis/kinetics_datasets/raw`, across all available organisms. Scope: peptidase BRENDA EC 3.4 entries and SABIO-RK entries containing `kcat`, `Km` or `kcat/Km`. A source entry is not necessarily an independent experiment.

- **Peptidases:** distinct reported UniProt accessions; EC numbers describe enzyme classes. Accessions do not distinguish all variants or mature chains. Missing accessions remain missing.
- **Contexts:** unique pairs of up to four residue tokens on either side of a resolved cut. The same context across enzymes or publications is counted once. Recognized nonstandard residues and D stereochemistry remain distinct; terminal labels are retained separately. Reporter half-contexts are a subset of partial contexts.
- **Full context:** P4–P1 / P1′–P4′, exactly four residues on each side. **Complete data:** at least one linked source entry with that full context, exactly one reported UniProt accession, an attributable numeric kinetic parameter with a known unit, a resolved publication, assay pH and temperature. It does not require all three kinetic parameter types.
- Cuts are resolved from explicit separators, a uniquely matching reported bond, exact substrate/product reconstruction, or a recognized terminal reporter linkage. Unknown chemistry, ambiguous positions and unsupported text remain unresolved. These counts measure recovered source-supported contexts; zero recovered contexts does not establish absence from the original literature. No predicted cuts or external sequence enrichment are used.

Raw files are read without modification. All processing stays in memory; no CSV or intermediate files are written. Expand the code cells to inspect the rules. `window_records` retains source locators and evidence for the recovered contexts.


In [1]:
from pathlib import Path
from dataclasses import dataclass
from functools import lru_cache
from itertools import permutations
import json
import math
import re
import tarfile

import ijson
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "assets/tools/proteolysis_data_manager").is_dir())
RAW = ROOT / "data/proteolysis/kinetics_datasets/raw"
REQUIRE_PH_TEMPERATURE = True
INCLUDE_NONSTANDARD = True
FAMILIES = {"turnover_number": "kcat", "km_value": "Km", "kcat_km_value": "kcat/Km"}
SI_UNITS = {"kcat": "s^(-1)", "Km": "M", "kcat/Km": "M^(-1)*s^(-1)"}
SOURCES = ("BRENDA", "SABIO-RK")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [2]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_NAMES = {
    "A": ("Ala", "alanine", "alanyl"), "R": ("Arg", "arginine", "arginyl"),
    "N": ("Asn", "asparagine", "asparaginyl"), "D": ("Asp", "aspartate", "aspartyl"),
    "C": ("Cys", "cysteine", "cysteinyl"), "Q": ("Gln", "glutamine", "glutaminyl"),
    "E": ("Glu", "glutamate", "glutamyl"), "G": ("Gly", "glycine", "glycyl"),
    "H": ("His", "histidine", "histidyl"), "I": ("Ile", "isoleucine", "isoleucyl"),
    "L": ("Leu", "leucine", "leucyl"), "K": ("Lys", "lysine", "lysyl"),
    "M": ("Met", "methionine", "methionyl"), "F": ("Phe", "phenylalanine", "phenylalanyl"),
    "P": ("Pro", "proline", "prolyl"), "S": ("Ser", "serine", "seryl"),
    "T": ("Thr", "threonine", "threonyl"), "W": ("Trp", "tryptophan", "tryptophyl"),
    "Y": ("Tyr", "tyrosine", "tyrosyl"), "V": ("Val", "valine", "valyl"),
}
RESIDUES = {name.lower(): symbol for symbol, names in AA_NAMES.items() for name in names}
RESIDUES.update({"nle": "Nle", "norleucine": "Nle", "nva": "Nva", "norvaline": "Nva",
                 "aib": "Aib", "abu": "Abu", "orn": "Orn", "ornithine": "Orn",
                 "4-nitrophenylalanine": "Phe(4-nitro)",
                 "(4-nitrophenylalanine)": "Phe(4-nitro)"})
REPORTERS = (
    "7-amido-4-carbamoylmethylcoumarin", "7-amido-4-fluoromethylcoumarin",
    "7-amino-4-methylcoumarin", "7-amido-4-methylcoumarin", "4-methylcoumarin-7-amide",
    "4-methyl-coumaryl-7-amide", "4-methoxy-2-naphthylamide", "2-naphthylamide",
    "p-nitroanilide", "4-nitroanilide", "nitroanilide", "pna", "amc", "afc", "acc", "mna",
)
N_CAPS = (
    "n-alpha-benzoyl", "n-benzyloxycarbonyl", "n-carbobenzoxy", "n-succinyl", "n-acetyl",
    "n-benzoyl", "t-butyloxycarbonyl", "tert-butyloxycarbonyl", "benzyloxycarbonyl",
    "(o-aminobenzoyl)", "o-aminobenzoyl", "2-aminobenzoyl", "aminobenzoyl",
    "succinyl", "acetyl", "benzoyl", "glutaryl", "dabcyl", "meosuc", "mocac",
    "fmoc", "abz", "cbz", "boc", "dnp", "mca", "suc", "glt", "tos", "ac", "bz", "z", "h",
)
C_CAPS = (*REPORTERS, "(N-(2,4-dinitrophenyl)ethylenediamine)",
          "N-(2,4-dinitrophenyl)ethylenediamine", "eddnp", "ednp", "edans", "dabcyl", "dnp", "nh2", "oh")
RESIDUE_PATTERN = re.compile("|".join(re.escape(name) for name in sorted(RESIDUES, key=len, reverse=True)), re.I)
UNIPROT = re.compile(r"(?<![A-Z0-9])(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})(?:-\d+)?(?![A-Z0-9])")
NUMBER = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"
POINT = re.compile(rf"({NUMBER})(?:\s*(?:±|\+/-)\s*({NUMBER}))?")
BOND_MOTIF = re.compile(r"\bcleavage\s+of\s+(?:the\s+)?([ACDEFGHIKLMNPQRSTVWYX])-([ACDEFGHIKLMNPQRSTVWYX])\s+bond\b")


@dataclass(frozen=True)
class Peptide:
    """Parsed source chemistry; unknown residue spellings are rejected."""
    residues: tuple
    n_cap: str | None
    c_cap: str | None
    reported_cut: int | None = None
    raw_label: str = ""


def accession_ids(values):
    """Extract reported UniProt accessions, including oligomer annotations.

    :param values: Original identifier strings or a list of strings.
    :return: Unique accessions; no identity is invented for missing identifiers.
    :rtype: frozenset
    """
    values = values if isinstance(values, list) else [values]
    return frozenset(hit for value in values if value for hit in UNIPROT.findall(str(value)))


def point_value(text):
    """Recognize a nonnegative finite point, keeping intervals unresolved.

    :param text: Original scalar or numeric text.
    :return: Whether the value is a supported numeric point.
    :rtype: bool
    """
    match = POINT.fullmatch(str(text).strip())
    return bool(match) and all(math.isfinite(float(x)) and float(x) >= 0
                               for x in match.groups() if x is not None)


def residues_from_text(text):
    """Parse the whole sequence spelling, without skipping unknown chemistry.

    :param str text: Sequence portion after recognized terminal labels.
    :return: Residue tokens, or null for unsupported/ambiguous syntax.
    :rtype: tuple | None
    """
    if re.fullmatch(rf"[{AA}]+", text) and (len(text) >= 4 or text.lower() not in RESIDUES):
        return tuple(text)
    if re.fullmatch(rf"[{AA}]+", text) and text.lower() in RESIDUES:
        return None  # Uppercase triplets can denote one residue or a short peptide.
    tokens, position = [], 0
    while position < len(text):
        stereo = re.match(r"([DL])-", text[position:])
        if stereo:
            position += stereo.end()
        match = RESIDUE_PATTERN.match(text, position)
        if match is None:
            return None
        token = RESIDUES[match.group().lower()]
        if stereo and stereo[1] == "D":
            token = "D-" + token
        if not INCLUDE_NONSTANDARD and token not in AA:
            return None
        tokens.append(token)
        position = match.end()
        if position == len(text):
            break
        separator = re.match(r"[-\s]+", text[position:])
        if separator is None:
            return None
        position += separator.end()
    return tuple(tokens) if tokens else None


@lru_cache(maxsize=None)
def parse_peptide(raw):
    """Parse a complete label, preserving recognized terminal groups and cuts.

    :param str raw: Original species name, comment or BRENDA substrate label.
    :return: Parsed source chemistry, or null if it cannot be resolved safely.
    :rtype: Peptide | None
    """
    body, n_cap, c_cap = raw.strip(), None, None
    for cap in sorted(N_CAPS, key=len, reverse=True):
        match = re.match(re.escape(cap) + r"[-\s]+", body, re.I)
        if match:
            n_cap, body = (None if cap == "h" else cap.lower()), body[match.end():]
            break
    for cap in sorted(C_CAPS, key=len, reverse=True):
        match = re.search(r"[-\s]+" + re.escape(cap) + r"$", body, re.I)
        if match:
            c_cap, body = (None if cap == "oh" else cap.lower()), body[:match.start()]
            break
    cut = None
    if body.count("/") == 1:
        left, right = body.split("/")
        if not re.fullmatch(rf"[{AA}]+", left) or not re.fullmatch(rf"[{AA}]+", right):
            return None
        tokens, cut = tuple(left + right), len(left)
    else:
        tokens = residues_from_text(body)
    return Peptide(tokens, n_cap, c_cap, cut, raw) if tokens else None


def species_peptide(species):
    """Require source name and comment to agree when both yield sequences.

    :param dict species: Original reaction participant.
    :return: One unambiguous parsed descriptor or null.
    :rtype: Peptide | None
    """
    descriptions = [(species.get("compound") or {}).get("name"), species.get("comment")]
    parsed = [p for text in descriptions if text and (p := parse_peptide(text)) is not None]
    if not parsed or len({p.residues for p in parsed}) != 1:
        return None
    return next((p for p in parsed if p.reported_cut is not None), parsed[0])


def resolved_sites(peptide, comment="", products=()):
    """Resolve cuts only from source notation, unique bond matches or products.

    :param Peptide peptide: Parsed substrate.
    :param str comment: Substrate-associated source comment.
    :param products: Parsed product descriptors from the same source reaction.
    :return: Position/evidence pairs; ambiguous candidate positions are omitted.
    :rtype: list
    """
    sites = {}
    if peptide.reported_cut is not None:
        sites[peptide.reported_cut] = "reported_separator"
    for left, right in BOND_MOTIF.findall(comment):
        positions = [i for i in range(1, len(peptide.residues))
                     if (left == "X" or peptide.residues[i - 1] == left)
                     and (right == "X" or peptide.residues[i] == right)]
        if len(positions) == 1:
            sites.setdefault(positions[0], "reported_bond_unique_match")
    candidates = set()
    if len(products) == 2:
        for left, right in permutations(products):
            if (left.residues + right.residues == peptide.residues
                    and left.n_cap == peptide.n_cap and right.c_cap == peptide.c_cap
                    and left.c_cap is None and right.n_cap is None):
                candidates.add(len(left.residues))
    if len(candidates) == 1:
        sites.setdefault(next(iter(candidates)), "reported_products_unique_match")
    if not sites and peptide.c_cap in {cap.lower() for cap in REPORTERS}:
        sites.setdefault(len(peptide.residues), "reporter_linkage")
    return sorted(sites.items())


def assay_scalar(comment, kind):
    """Recognize one explicit assay condition, rejecting ranges and optima.

    :param str comment: Original BRENDA kinetic comment.
    :param str kind: ``ph`` or ``temperature``.
    :return: Whether a supported scalar condition is present.
    :rtype: bool
    """
    if re.search(r"\b(?:optimum|optimal|stability|storage|stored|expression|purification)\b", comment, re.I):
        return False
    pattern = r"\bpH\s*(?:=|:|of|at)?\s*(\d+(?:\.\d+)?)" if kind == "ph" else r"(?<![\w.−–-])(\d+(?:\.\d+)?)\s*°\s*C\b"
    matches = list(re.finditer(pattern, comment, re.I))
    if len(matches) != 1:
        return False
    hit = matches[0]
    before, after = comment[max(0, hit.start() - 35):hit.start()], comment[hit.end():hit.end() + 25]
    if re.search(r"(?:[<>≤≥~≈]|\b(?:between|to|and|or|below|above|about))\s*$", before, re.I):
        return False
    if re.match(r"\s*(?:[-–±/]|to\b|and\s+\d|or\s+\d)", after, re.I):
        return False
    return 0 <= float(hit[1]) <= 14 if kind == "ph" else True


def has_publication(reference):
    """Accept a resolved bibliographic record rather than an unexpanded ID.

    :param dict reference: Original source publication dictionary.
    :return: Whether bibliographic information is present.
    :rtype: bool
    """
    return bool(reference.get("pmid") or reference.get("pubmed_id") or reference.get("title"))


def add_windows(source, record_id, peptide, sites, enzyme_ids, numeric, publication, ph, temperature):
    """Retain one source association per resolved site without cloning rates.

    :param str source: Source database.
    :param str record_id: Original record locator.
    :param Peptide peptide: Parsed original substrate chemistry.
    :param sites: Resolved position/evidence pairs.
    :param enzyme_ids: Reported UniProt IDs.
    :param bool numeric: A numeric kinetic parameter attributable to the substrate.
    :param bool publication: A linked publication is available.
    :param bool ph: An assay pH is recorded.
    :param bool temperature: An assay temperature is recorded.
    """
    for position, evidence in sites:
        left = peptide.residues[max(0, position - 4):position]
        right = peptide.residues[position:position + 4]
        full = len(left) == len(right) == 4
        attributable = numeric and len(sites) == 1
        complete = full and attributable and len(enzyme_ids) == 1 and publication
        complete = complete and (not REQUIRE_PH_TEMPERATURE or (ph and temperature))
        windows.append({"source": source, "source_record_id": record_id,
                        "context": (left, right), "sequence_tokens": peptide.residues,
                        "substrate_raw": peptide.raw_label, "n_cap": peptide.n_cap, "c_cap": peptide.c_cap,
                        "cut_after": position, "full_8_residues": full,
                        "reporter": evidence == "reporter_linkage", "evidence": evidence,
                        "enzyme_ids": enzyme_ids, "numeric": attributable, "complete_data": complete})

In [3]:
manifest = json.loads((RAW / "manifest.json").read_text())
windows = []
source_stats = {source: {"records": 0, "ec": set(), "enzyme_ids": set(), "without_id": 0} for source in SOURCES}
brenda_filename = next(iter(manifest["sources"]["brenda"]["files"]))
with tarfile.open(RAW / "brenda" / brenda_filename) as archive:
    members = [m for m in archive.getmembers() if m.isfile() and m.name.endswith(".json")]
    if len(members) != 1:
        raise ValueError("Expected one BRENDA JSON member")
    with archive.extractfile(members[0]) as handle:
        for ec, block in ijson.kvitems(handle, "data"):
            if not ec.startswith("3.4."):
                continue
            protein_ids = {str(key): accession_ids(value.get("accessions") or [])
                           for key, value in (block.get("protein") or {}).items()}
            references = block.get("reference") or {}
            for family in FAMILIES:
                for index, entry in enumerate(block.get(family) or []):
                    ids = frozenset(identifier for key in entry.get("proteins") or []
                                    for identifier in protein_ids.get(str(key), ()))
                    stats = source_stats["BRENDA"]
                    stats["records"] += 1
                    stats["ec"].add(ec)
                    stats["enzyme_ids"].update(ids)
                    stats["without_id"] += not bool(ids)
                    match = re.fullmatch(r"\s*([^{}]*?)\s*(?:\{(.*)\})?\s*", entry.get("value") or "", re.S)
                    label = (match[2] or "").strip() if match else ""
                    peptide = parse_peptide(label)
                    if peptide is None:
                        continue
                    comment = entry.get("comment") or ""
                    sites = resolved_sites(peptide, comment)
                    numeric = bool(match) and point_value(match[1])
                    publication = any(has_publication(references.get(str(key)) or {})
                                      for key in entry.get("references") or [])
                    add_windows("BRENDA", f"data/{ec}/{family}/{index}", peptide, sites, ids, numeric,
                                publication, assay_scalar(comment, "ph"), assay_scalar(comment, "temperature"))

sabio_records = {}
for filename in sorted(manifest["sources"]["sabio"]["files"]):
    for record in json.loads((RAW / "sabio" / filename).read_text())["data"]:
        if record["id"] in sabio_records:
            raise ValueError("Duplicate SABIO entry ID")
        sabio_records[record["id"]] = record
for entry_id, record in sabio_records.items():
    parameters = [p for p in (record.get("kineticlaw") or {}).get("parameter") or []
                  if (p.get("parameter_type") or {}).get("name") in SI_UNITS]
    if not parameters:
        continue
    enzyme = record.get("enzyme_description") or {}
    ids = frozenset(identifier for protein in enzyme.get("proteins") or []
                    for identifier in accession_ids(protein.get("uniprot_id")))
    stats = source_stats["SABIO-RK"]
    stats["records"] += 1
    if enzyme.get("ec_number"):
        stats["ec"].add(enzyme["ec_number"])
    stats["enzyme_ids"].update(ids)
    stats["without_id"] += not bool(ids)
    species = (record.get("reaction") or {}).get("species") or []
    substrates = [(i, s) for i, s in enumerate(species) if s.get("role") == "Substrate"
                  and (s.get("compound") or {}).get("name") not in {"H2O", "H+", "Water"}]
    products = [p for s in species if s.get("role") == "Product" and (p := species_peptide(s)) is not None]
    conditions = record.get("experimental_conditions") or {}
    ph_info, temperature_info = conditions.get("envvar_ph") or {}, conditions.get("envvar_temperature") or {}
    ph = (point_value(ph_info.get("start_value")) and 0 <= float(ph_info["start_value"]) <= 14
          and ph_info.get("end_value") in (None, ph_info.get("start_value")))
    temperature = (temperature_info.get("start_value") is not None
                   and math.isfinite(float(temperature_info["start_value"]))
                   and temperature_info.get("end_value") in (None, temperature_info.get("start_value"))
                   and temperature_info.get("unit") in {"°C", "K", "°F", "Celsius", "Kelvin", "Fahrenheit"})
    for index, substrate in substrates:
        peptide = species_peptide(substrate)
        if peptide is None:
            continue
        numeric = False
        name = (substrate.get("compound") or {}).get("name")
        for parameter in parameters:
            kind = parameter["parameter_type"]["name"]
            unit = parameter.get("unit") or {}
            value, end = parameter.get("n_start_value"), parameter.get("n_end_value")
            valid = unit.get("n_name") == SI_UNITS[kind] and point_value(value) and end in (None, value)
            target = ((parameter.get("species") or {}).get("species_key") or "").split(" | ")
            explicit = (len(target) == 3 and target[1] == name and target[2] == "Substrate"
                        and sum((s.get("compound") or {}).get("name") == name for _, s in substrates) == 1)
            numeric |= valid and (explicit or len(substrates) == 1)
        sites = resolved_sites(peptide, substrate.get("comment") or "", products)
        add_windows("SABIO-RK", f"entry/{entry_id}/reaction/species/{index}", peptide, sites, ids, numeric,
                    has_publication(record.get("publication") or {}), ph, temperature)

In [4]:
window_records = pd.DataFrame(windows)
peptidase_rows, context_rows = [], []
for source in SOURCES:
    stats = source_stats[source]
    rows = [row for row in windows if row["source"] == source]
    peptidase_rows.append({
        "Source": source, "Kinetic source entries": stats["records"],
        "UniProt IDs": len(stats["enzyme_ids"]), "EC numbers": len(stats["ec"]),
        "UniProt IDs with context": len({identifier for row in rows for identifier in row["enzyme_ids"]}),
        "Entries without UniProt ID": stats["without_id"],
    })
    context_rows.append({
        "Source": source,
        "Unique contexts": len({row["context"] for row in rows}),
        "Full 8-residue contexts": len({row["context"] for row in rows if row["full_8_residues"]}),
        "Partial contexts": len({row["context"] for row in rows if not row["full_8_residues"]}),
        "Reporter half-contexts": len({row["context"] for row in rows if row["reporter"]}),
        "Full contexts + numeric kinetics": len({row["context"] for row in rows if row["full_8_residues"] and row["numeric"]}),
        "Full contexts + complete data": len({row["context"] for row in rows if row["complete_data"]}),
    })
peptidases = pd.DataFrame(peptidase_rows).set_index("Source")
contexts = pd.DataFrame(context_rows).set_index("Source")

In [5]:
display(peptidases)
display(contexts)

,Kinetic source entries,UniProt IDs,EC numbers,UniProt IDs with context,Entries without UniProt ID
Source,,,,,
BRENDA,26697,451,327,178,20182
SABIO-RK,2156,258,115,130,210


,Unique contexts,Full 8-residue contexts,Partial contexts,Reporter half-contexts,Full contexts + numeric kinetics,Full contexts + complete data
Source,,,,,,
BRENDA,413,0,413,342,0,0
SABIO-RK,300,40,260,70,40,29


## Common numerical parameters

`common_parameters` has numeric kcat (s⁻¹), km (M) and kcat_km (M⁻¹ s⁻¹) columns alongside P4–P4′, source, enzyme and substrate. kcat/km is derived only for a unique source-assay/substrate pair without a reported efficiency. Examples below show each parameter with an actual linked window. A linked substrate-level rate is not automatically a rate for one specific cut. All transformations stay in memory; no CSV or database is written.

In [6]:
# Use the independent data library from this checkout.
import sys
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
from proteolysis_data_manager.integration.merge import (
    build_dataset, parse_dataset, add_merops_scores,
)
contract = build_dataset(ROOT / "data/proteolysis", sources=["brenda", "sabio"])
common_parameters = contract.to_frame()
# Show each numerical parameter together with its actual linked cut window.
examples = pd.concat([
    common_parameters.loc[common_parameters[parameter].notna() & common_parameters.has_window]
    .groupby("source_db", sort=False).head(2)
    for parameter in ("kcat", "km", "kcat_km")
]).drop_duplicates("row_id")
display(examples[[
    "source_db", "enzyme_name", "substrate_name", "window",
    "kcat", "km", "kcat_km", "kcat_km_basis", "ph", "temperature_c", "site_evidence",
]])


Retained 2 descending BRENDA ranges as unresolved source values


,source_db,enzyme_name,substrate_name,window,kcat,km,kcat_km,kcat_km_basis,ph,temperature_c,site_evidence
0,brenda,leucyl aminopeptidase,L-Leu-4-nitroanilide,- - - L | - - - -,0.700,NaN,NaN,None,NaN,NaN,reporter_linkage
1,brenda,leucyl aminopeptidase,L-Ala-4-nitroanilide,- - - A | - - - -,0.078,NaN,NaN,None,7.5,80.0,reporter_linkage
26697,sabio,subtilisin,Peptide,A A P F | - - - -,220.000,0.00190,120000.0,reported,7.5,25.0,reporter_linkage
26698,sabio,subtilisin,Peptide,A A P F | - - - -,200.000,0.00340,5800000.0,reported,7.5,25.0,reporter_linkage
161,brenda,leucyl aminopeptidase,leucine-7-amido-4-methylcoumarin,- - - L | - - - -,NaN,0.00001,NaN,None,NaN,NaN,reporter_linkage
163,brenda,leucyl aminopeptidase,L-Leu-4-nitroanilide,- - - L | - - - -,NaN,0.00110,NaN,None,NaN,NaN,reporter_linkage
370,brenda,leucyl aminopeptidase,L-Leu-4-nitroanilide,- - - L | - - - -,NaN,NaN,2.5,reported,7.5,35.0,reporter_linkage
371,brenda,leucyl aminopeptidase,L-Ala-4-nitroanilide,- - - A | - - - -,NaN,NaN,3.2,reported,7.5,80.0,reporter_linkage
